<a href="https://colab.research.google.com/github/Violetaaa/tfg-learning/blob/main/mvp_agente_gis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP: agente GIS para acceso a servicios

**Para qué sirve este cuaderno:** ver la arquitectura completa funcionando de punta a punta, con datos reales de un concejo.
No busca buenos resultados todavía, sino entender cómo encajan las piezas antes de escribir la PEC1.

```
Pregunta ─► AGENTE (LangGraph: llm ⇄ action) ─► HERRAMIENTAS GIS (Python) ─► DATOS (OSM, caché local)
                                                        ▲
EVALUACIÓN: respuesta de referencia calculada con las mismas herramientas, sin LLM
```

**La idea clave:** el LLM nunca calcula nada. Solo decide **qué** herramientas llamar, **en qué orden** y **con qué argumentos**.
Los números salen siempre de Python. Así, si la respuesta está mal, sabemos que el fallo es del razonamiento del agente y no de los datos.

Capas (una sección por capa):
1. **Datos:** núcleos y servicios de un concejo descargados de OpenStreetMap y guardados en JSON.
2. **Herramientas GIS:** funciones Python deterministas. No saben nada del LLM y se pueden probar solas.
3. **Agente:** el mismo grafo del curso (`llm` → `action` → `llm`). Funciona con **Gemini** (API) o con un **modelo local** (Ollama) cambiando una línea.
4. **Evaluación:** pocas preguntas, cada una con su respuesta de referencia calculada por código y su plan esperado.

**Cómo ejecutarlo:**
- **En Colab con Gemini:** sube el cuaderno, pon `PROVEEDOR = "gemini"` en la sección 3 y ejecuta todo. Necesita `GOOGLE_API_KEY` en Secrets.
- **En tu Mac con Ollama:** ten Ollama arrancado y el modelo descargado (`ollama pull qwen3.8`), abre el cuaderno con Jupyter, pon `PROVEEDOR = "ollama"` y ejecuta todo.

Simplificaciones del MVP (a sustituir en el TFG):
- El **tiempo en coche es una estimación**: distancia en línea recta × 1,4 a 40 km/h. No es un cálculo por carretera; en el TFG se sustituye por OSMnx precalculado.
- Un solo concejo. Los servicios se buscan con un margen de ~10 km alrededor, porque el más cercano puede estar en el concejo vecino.
- OSM no tiene la población de la mayoría de núcleos, así que aquí no hay herramienta de población. En el TFG vendrá del nomenclátor de SADEI.

In [3]:
# Install dependencies. %pip (not !pip) installs into the kernel this notebook is using,
# so this cell works both in Colab and in a local Jupyter / virtualenv.
%pip install -q langgraph langchain-google-genai langchain-ollama requests

In [13]:
import os, json, math, time, requests
from typing import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

# Study area. Change CONCEJO to try another municipality (the data cell will download it).
CONCEJO = "Candamo"
# Extra margin around the municipality, in degrees (~10 km), so that services just across
# the border (often the nearest ones in rural areas) are also included.
MARGEN_GRADOS = 0.1

## 1. Datos (OSM → JSON)

**Qué hace:** pide a OpenStreetMap (a través de la API Overpass) los núcleos del concejo y los servicios de alrededor,
y los guarda en un JSON para no volver a descargarlos.

**Por qué así:** el agente debe trabajar sobre datos fijos y conocidos. Si cada ejecución consultara internet,
los resultados cambiarían y no se podrían comparar. El JSON es la "foto" de los datos con la que se evalúa.

**También es la prueba de datos para la PEC1:** al final imprime cuántos servicios hay de cada tipo.
Si salen 0 o muy pocos, es un riesgo que hay que apuntar.

In [15]:
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]
HEADERS = {"User-Agent": "TFG-UOC-agente-gis/0.1 (academic project)"}

# Translation table: OSM tags -> our service types.
# Health facilities are tagged inconsistently in OSM (clinic, doctors, healthcare=centre...),
# so several tags map to "centro_salud". This mapping is a modelling decision that must be
# checked against the official catalogue (Astursalud) before trusting it.
TIPOS = {
    "farmacia":     [("amenity", "pharmacy")],
    "centro_salud": [("amenity", "clinic"), ("amenity", "doctors"), ("healthcare", "centre"), ("healthcare", "doctor")],
    "hospital":     [("amenity", "hospital")],
    "colegio":      [("amenity", "school")],
}

def overpass(query):
    errores = []
    for url in OVERPASS_URLS:
        try:
            r = requests.post(url, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            return r.json()["elements"]
        except Exception as e:
            errores.append(f"{url}: {e}")
            time.sleep(2)
    raise RuntimeError("Ningún servidor Overpass respondió:\n" + "\n".join(errores))

def coords(el):
    """OSM points (nodes) have lat/lon; buildings and areas (ways/relations) come with a 'center'."""
    if "lat" in el:
        return el["lat"], el["lon"]
    return el["center"]["lat"], el["center"]["lon"]

def descargar(concejo):
    # Step 1: settlements inside the municipality.
    # admin_level=8 is the municipality level in Spain; place=town/village/hamlet are villa/pueblo/aldea.
    q_nucleos = f"""[out:json][timeout:90];
    area["name"="{concejo}"]["admin_level"="8"]->.a;
    node["place"~"^(town|village|hamlet)$"](area.a);
    out body;"""   # "out body" returns coordinates + tags ("out tags" would return tags only)
    nucleos = []
    for el in overpass(q_nucleos):
        lat, lon = coords(el)
        nucleos.append({"id": f"n{el['id']}", "nombre": el["tags"].get("name", "?"),
                        "tipo": el["tags"]["place"], "lat": lat, "lon": lon, "concejo": concejo})

    # Step 2: services inside the rectangle that contains all settlements, plus the margin.
    # A rectangle (not the municipal border) is used on purpose: see MARGEN_GRADOS above.
    s = min(n["lat"] for n in nucleos) - MARGEN_GRADOS
    n_ = max(n["lat"] for n in nucleos) + MARGEN_GRADOS
    w = min(n["lon"] for n in nucleos) - MARGEN_GRADOS
    e = max(n["lon"] for n in nucleos) + MARGEN_GRADOS
    filtros = "".join(f'nwr["{k}"="{v}"]({s},{w},{n_},{e});'
                      for pares in TIPOS.values() for k, v in pares)
    servicios = []
    # "out center": nodes keep their lat/lon; buildings and areas get a single center point
    for el in overpass(f"[out:json][timeout:90];({filtros});out center;"):
        tags = el.get("tags", {})
        for tipo, pares in TIPOS.items():
            if any(tags.get(k) == v for k, v in pares):
                lat, lon = coords(el)
                servicios.append({"id": f"s{el['id']}", "tipo": tipo,
                                  "nombre": tags.get("name", f"({tipo} sin nombre)"),
                                  "lat": lat, "lon": lon})
                break   # each element gets one type only
    return {"nucleos": nucleos, "servicios": servicios}

# Cache: download once, then always reuse the same file (reproducibility).
# Delete the JSON file if you want to download fresh data.
FICHERO = f"datos_{CONCEJO.replace(' ', '_')}.json"
if os.path.exists(FICHERO):
    DATOS = json.load(open(FICHERO))
else:
    DATOS = descargar(CONCEJO)
    json.dump(DATOS, open(FICHERO, "w"), ensure_ascii=False)

# Data check: how much did OSM give us? Compare with what you know about the area.
print(f"Núcleos en {CONCEJO}: {len(DATOS['nucleos'])}")
for tipo in TIPOS:
    print(f"  {tipo:13s}: {sum(s['tipo'] == tipo for s in DATOS['servicios'])} ")

Núcleos en Candamo: 54
  farmacia     : 53 
  centro_salud : 34 
  hospital     : 4 
  colegio      : 67 


## 2. Herramientas GIS

**Qué hace:** define las operaciones que el agente puede usar: buscar un lugar, buscar los servicios más cercanos
y listar los núcleos de un concejo.

**Por qué en dos niveles:**
- **Funciones internas** (`_buscar_lugar`, `_servicios_cercanos`…): Python normal. Las usa la **evaluación** para calcular la respuesta correcta.
- **Herramientas** (`@tool buscar_lugar`…): envuelven las anteriores para el **agente**. La docstring es lo que lee el LLM para decidir cuándo usar cada una.

Como agente y evaluación usan el mismo código y los mismos datos, una diferencia entre ambos solo puede venir del agente.

**Por qué herramientas genéricas** (y no una por tipo de pregunta): si existiera `farmacia_mas_cercana_a(pueblo)`,
el LLM no tendría nada que planificar. Con piezas pequeñas tiene que decidir cómo combinarlas, que es lo que se quiere estudiar.

In [16]:
def distancia_km(a, b):
    """Great-circle (straight-line) distance between two points with lat/lon, using the Haversine formula."""
    R = 6371  # Earth radius in km
    p1, p2 = math.radians(a["lat"]), math.radians(b["lat"])
    dp, dl = p2 - p1, math.radians(b["lon"] - a["lon"])
    h = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))

def minutos_estimados(a, b):
    """MVP placeholder for driving time: straight line x1.4 (to approximate road curves) at 40 km/h.
    It is NOT a road-network time. In the TFG this is replaced by OSMnx travel times."""
    return round(distancia_km(a, b) * 1.4 / 40 * 60, 1)

# --- Internal functions: plain Python, used by the evaluation and wrapped by the tools ---

def _buscar_lugar(nombre):
    """All settlements whose name matches exactly (case-insensitive). More than one = ambiguous name."""
    nombre = nombre.strip().lower()
    return [n for n in DATOS["nucleos"] if n["nombre"].lower() == nombre]

def _nucleo(nucleo_id):
    return next(n for n in DATOS["nucleos"] if n["id"] == nucleo_id)

def _servicios_cercanos(nucleo_id, tipo, k=3):
    """The k services of a given type closest to a settlement, sorted by straight-line distance."""
    origen = _nucleo(nucleo_id)
    candidatos = [s for s in DATOS["servicios"] if s["tipo"] == tipo]
    candidatos.sort(key=lambda s: distancia_km(origen, s))
    return [{"id": s["id"], "nombre": s["nombre"],
             "km_linea_recta": round(distancia_km(origen, s), 1),
             "minutos_estimados": minutos_estimados(origen, s)} for s in candidatos[:k]]

def _nucleos_en(concejo):
    return [{"id": n["id"], "nombre": n["nombre"], "tipo": n["tipo"]}
            for n in DATOS["nucleos"] if n["concejo"].lower() == concejo.strip().lower()]

# --- Tools for the LLM ---
# The docstrings are written in Spanish on purpose: they are the "instructions" the model reads.
# Tools return JSON text (the LLM reads text) and return errors as text instead of raising,
# so the agent can read the error and correct itself instead of crashing the whole run.

@tool
def buscar_lugar(nombre: str) -> str:
    """Busca un núcleo de población por su nombre exacto. Devuelve una lista de candidatos con id,
    tipo y concejo. Si hay varios candidatos, el nombre es ambiguo. Si la lista está vacía, no existe en los datos."""
    return json.dumps([{k: n[k] for k in ("id", "nombre", "tipo", "concejo")} for n in _buscar_lugar(nombre)],
                      ensure_ascii=False)

@tool
def servicios_cercanos(nucleo_id: str, tipo: str, k: int = 3) -> str:
    """Devuelve los k servicios más cercanos a un núcleo (usa el id de buscar_lugar o nucleos_en).
    tipo: 'farmacia', 'centro_salud', 'hospital' o 'colegio'.
    Incluye distancia en línea recta y minutos estimados en coche."""
    if tipo not in TIPOS:
        return f"Tipo desconocido: {tipo}. Tipos válidos: {list(TIPOS)}"
    if not any(n["id"] == nucleo_id for n in DATOS["nucleos"]):
        return f"No existe el núcleo con id {nucleo_id}. Usa buscar_lugar o nucleos_en para obtener el id."
    return json.dumps(_servicios_cercanos(nucleo_id, tipo, k), ensure_ascii=False)

@tool
def nucleos_en(concejo: str) -> str:
    """Lista los núcleos de población (id, nombre, tipo) de un concejo de Asturias."""
    return json.dumps(_nucleos_en(concejo), ensure_ascii=False)

TOOLS = [buscar_lugar, servicios_cercanos, nucleos_en]

# Sanity check without any LLM: if this prints something sensible, the GIS layer works on its own.
ejemplo = DATOS["nucleos"][0]
print(ejemplo["nombre"], "->", _servicios_cercanos(ejemplo["id"], "farmacia", 1))

Grullos -> [{'id': 's8024694974', 'nombre': '(farmacia sin nombre)', 'km_linea_recta': 2.1, 'minutos_estimados': 4.4}]


## 3. Modelo: Gemini o local

**Solo hay que tocar `PROVEEDOR`.** Lo demás del cuaderno no cambia, porque LangChain da la misma interfaz
(`invoke`, `bind_tools`) a los dos modelos.

| | Gemini (API) | Ollama (local) |
|---|---|---|
| Dónde | Colab o tu Mac | Solo tu Mac (Colab no ve tu ordenador) |
| Coste | Gratis con límite (~20 peticiones/día por modelo) | Gratis, sin límite |
| Velocidad | Rápido | Más lento (depende del tamaño del modelo) |
| Para el TFG | Modelo comercial | Modelo abierto |

Poder cambiar de modelo sin tocar nada más es lo que permite, en el TFG, comparar el mismo agente con modelos distintos.

In [38]:
PROVEEDOR = "gemini"   # "gemini" or "ollama"

"""
Modelos disponibles con la API de Gemini:
models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview
"""

MODELOS = {
    "gemini": "gemini-3.1-flash-lite",        # Modelo de última generación activo en la lista oficial
    "ollama": "qwen3.8",                 # must be pulled first: `ollama pull qwen3.8`
}

if PROVEEDOR == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    # The API key comes from Colab Secrets when running in Colab,
    # or from the GOOGLE_API_KEY environment variable when running locally.
    if "GOOGLE_API_KEY" not in os.environ:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    # temperature=0: as deterministic as possible, so repeated runs are comparable.
    model = ChatGoogleGenerativeAI(model=MODELOS["gemini"], temperature=0)

elif PROVEEDOR == "ollama":
    from langchain_ollama import ChatOllama
    # Ollama must be running on this machine (`brew services start ollama`).
    # Some models (e.g. qwen3.8) "think" before answering by default, which makes every step slower.
    # reasoning=False turns that off for the MVP. Set it to True to compare with thinking enabled.
    model = ChatOllama(model=MODELOS["ollama"], temperature=0, reasoning=False)

else:
    raise ValueError('PROVEEDOR must be "gemini" or "ollama"')

print("Usando:", PROVEEDOR, "-", MODELOS["gemini"])

Usando: gemini - gemini-3.1-flash-lite


## 4. Agente (el grafo del curso)

**Qué hace:** es el mismo patrón de la lección 2 del curso:
el nodo `llm` decide → si pide herramientas, el nodo `action` las ejecuta → vuelta a `llm` → … hasta que responde sin pedir más.

**Qué se añade respecto al curso:**
- Un **prompt de sistema** que obliga a usar las herramientas y a decir cuándo no se puede responder.
- La **traza**: la lista de herramientas que llamó, en orden y con sus argumentos. Es lo que permite evaluar *cómo* razona el agente, no solo *qué* responde.
- Un **límite de pasos**, para que un modelo que se queda en bucle no se ejecute indefinidamente.

In [39]:
class AgentState(TypedDict):
    # The whole state is the list of messages. operator.add means each node's output
    # is appended to the list instead of replacing it (that is how the conversation grows).
    messages: Annotated[list[AnyMessage], operator.add]

# System prompt: the rules of behaviour. Each rule targets a failure we want to measure:
# using memory instead of tools, guessing on ambiguous places, inventing unavailable data.
SYSTEM = """Eres un asistente que responde preguntas sobre acceso a servicios en la Asturias rural.
Usa SIEMPRE las herramientas para obtener datos; no uses lo que sepas de memoria.
Si un lugar es ambiguo o no existe en los datos, dilo en vez de suponer.
Si la pregunta pide algo que las herramientas no pueden calcular (p. ej. transporte público), dilo.
Los minutos son estimaciones; indícalo en la respuesta. Responde en español."""

class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)           # the model decides
        graph.add_node("action", self.take_action)     # Python executes
        # After "llm": if the model asked for tools go to "action", otherwise finish.
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")                # tool results always go back to the model
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        # bind_tools sends the tool names, arguments and docstrings to the model on every call
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        # The model "asks" for a tool by returning tool_calls instead of (or as well as) text
        return len(state["messages"][-1].tool_calls) > 0

    def call_llm(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        return {"messages": [self.model.invoke(messages)]}

    def take_action(self, state: AgentState):
        results = []
        for t in state["messages"][-1].tool_calls:
            if t["name"] not in self.tools:
                result = "bad tool name, retry"        # the model invented a tool: tell it, don't crash
            else:
                result = self.tools[t["name"]].invoke(t["args"])
            # tool_call_id links each result to the request that produced it
            results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result)))
        return {"messages": results}

def preguntar(agent, pregunta, max_pasos=15):
    """Run one question. Returns the final text, the trace of tool calls and the time it took."""
    inicio = time.time()
    # recursion_limit caps the number of graph steps (llm and action each count as one)
    out = agent.graph.invoke({"messages": [HumanMessage(content=pregunta)]},
                             {"recursion_limit": max_pasos})
    segundos = round(time.time() - inicio, 1)
    # Trace = every tool call the model made, in order: [(tool_name, arguments), ...]
    traza = [(tc["name"], tc["args"]) for m in out["messages"] for tc in getattr(m, "tool_calls", [])]
    final = out["messages"][-1]
    # .text joins text blocks (Gemini returns content as a list of blocks; .content would show them raw)
    texto = final.text if hasattr(final, "text") else final.content
    return texto, traza, segundos

abot = Agent(model, TOOLS, system=SYSTEM)

## 5. Evaluación mínima

**Qué hace:** lanza 3 preguntas y, para cada una, compara lo que hizo el agente con lo que debería haber hecho.

**Por qué así (es lo que pidió el tutor, evaluación objetiva):**
- La **referencia** se calcula con las funciones internas, sin LLM. Es la respuesta correcta respecto a nuestros datos.
- El **plan esperado** es la secuencia de herramientas que debería usar. Permite evaluar el razonamiento, no solo el resultado.
- Hay una **pregunta trampa** (autobús) que no se puede responder con nuestros datos. Mide si el agente sabe decir "no lo sé" en vez de inventar.

Las preguntas se generan a partir de un núcleo real de los datos descargados, para que siempre tengan sentido.

**Coste:** con Gemini, cada pregunta gasta 2–5 peticiones, así que unas 10–15 de tus 20 diarias. Con Ollama no hay límite, pero tarda más.
El tiempo de cada pregunta se imprime para poder comparar modelos.

In [40]:
# Choose a settlement whose name is unique (so the question is not ambiguous by accident),
# preferring a village over a hamlet.
unicos = [n for n in DATOS["nucleos"] if len(_buscar_lugar(n["nombre"])) == 1]
NUCLEO = next((n for n in unicos if n["tipo"] == "village"), unicos[0])
ref_farmacia = _servicios_cercanos(NUCLEO["id"], "farmacia", 1)[0]

PREGUNTAS = [
    {   # Level 1: a 2-step chain (find the place, then search around it)
        "pregunta": f"¿Cuál es la farmacia más cercana a {NUCLEO['nombre']}?",
        "referencia": ref_farmacia["nombre"],
        "plan_esperado": ["buscar_lugar", "servicios_cercanos"],
        "auto": True,   # the reference is a name, so checking whether it appears in the text is meaningful
    },
    {   # Level 2: two searches and a comparison
        "pregunta": f"Desde {NUCLEO['nombre']}, ¿qué está más cerca, el centro de salud o el colegio más cercano?",
        "referencia": min(["centro_salud", "colegio"],
                          key=lambda t: (_servicios_cercanos(NUCLEO["id"], t, 1) or [{"km_linea_recta": 1e9}])[0]["km_linea_recta"]),
        "plan_esperado": ["buscar_lugar", "servicios_cercanos", "servicios_cercanos"],
        # Not automatic: both words will appear in the answer, so a text match proves nothing
    },
    {   # Trap: there is no public transport data, the right behaviour is to say so
        "pregunta": f"¿Cuánto se tarda en autobús desde {NUCLEO['nombre']} al hospital?",
        "referencia": "debe decir que no hay datos de autobús",
        "plan_esperado": [],
    },
]

print("Modelo:", PROVEEDOR, "-", MODELOS[PROVEEDOR])
for i, p in enumerate(PREGUNTAS):
    if i > 0:
        # Sleep between queries to avoid hitting the Gemini Free Tier Rate Limits (Max 5 RPM)
        print("\n⏳ Esperando 10 segundos para respetar los límites de cuota de la API...")
        time.sleep(10)

    reintentos = 3
    for intento in range(reintentos):
        try:
            texto, traza, segundos = preguntar(abot, p["pregunta"])
            break
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print(f"⚠️ Límite de cuota alcanzado. Esperando 45 segundos para reintentar (intento {intento+1}/{reintentos})...")
                time.sleep(45)
                if intento == reintentos - 1:
                    raise e
            else:
                raise e

    usadas = [nombre for nombre, _ in traza]
    print("=" * 80)
    print("PREGUNTA:  ", p["pregunta"])
    print("TRAZA:     ", traza)                                   # what the agent actually did
    print("PLAN OK:   ", usadas == p["plan_esperado"], "| esperado:", p["plan_esperado"])
    print("REFERENCIA:", p["referencia"])                         # what it should have answered
    print("RESPUESTA: ", texto)
    print("TIEMPO:    ", segundos, "s")
    if p.get("auto"):
        print("ACIERTO (automático, aproximado):", p["referencia"].lower() in texto.lower())
    else:
        print("ACIERTO: revisar a mano")

Modelo: gemini - gemini-3.1-flash-lite
PREGUNTA:   ¿Cuál es la farmacia más cercana a Grullos?
TRAZA:      [('buscar_lugar', {'nombre': 'Grullos'}), ('servicios_cercanos', {'tipo': 'farmacia', 'k': 1, 'nucleo_id': 'n32847081'})]
PLAN OK:    True | esperado: ['buscar_lugar', 'servicios_cercanos']
REFERENCIA: (farmacia sin nombre)
RESPUESTA:  La farmacia más cercana a Grullos se encuentra a una distancia de 2,1 km en línea recta, lo que supone un tiempo estimado de unos 4 o 5 minutos en coche. Ten en cuenta que los datos disponibles no especifican el nombre del establecimiento.
TIEMPO:     19.0 s
ACIERTO (automático, aproximado): False

⏳ Esperando 10 segundos para respetar los límites de cuota de la API...
PREGUNTA:   Desde Grullos, ¿qué está más cerca, el centro de salud o el colegio más cercano?
TRAZA:      [('buscar_lugar', {'nombre': 'Grullos'}), ('servicios_cercanos', {'nucleo_id': 'n32847081', 'k': 1, 'tipo': 'centro_salud'}), ('servicios_cercanos', {'k': 1, 'nucleo_id': 'n3284708

## Qué mirar después de ejecutarlo

- **Datos:** ¿cuántos centros de salud y farmacias salieron? ¿Coinciden con lo que conoces de la zona? Si no, es un riesgo para la PEC1.
- **Traza:** ¿el agente siguió el plan esperado o dio pasos de más o de menos?
- **Trampa:** ¿dijo que no hay datos de autobús o se inventó un tiempo?
- **Gemini frente a local:** ejecuta el cuaderno con cada `PROVEEDOR` y compara trazas, aciertos y tiempos. Es una versión mínima de la comparación del TFG.
- **Acierto automático:** comparar texto es frágil. En el TFG conviene pedir al agente una respuesta estructurada (JSON) además del texto.

Qué cambia en el TFG respecto a este MVP: tiempos reales por carretera (OSMnx precalculado), población de SADEI,
más herramientas (`tiempo_viaje`, `poblacion`), más preguntas y, si se decide, una segunda estrategia (planificar y ejecutar) sobre las mismas herramientas.